# ☁️ Google Colab Training Template — Cart Abandonment Tabular ML Suite

> **Sprint 1 (Task 1.11) → Sprint 2 Execution**  
> Standardized notebook template for training **LightGBM (`IntentClassifier`)**, **XGBoost (`ChurnPredictor`)**, **CausalML X-Learner (`UpliftModel`)**, and **Lifelines CoxPH (`TimingOptimizer`)** on Google Colab with Google Drive storage and MLflow experiment tracking.


In [ ]:
# 1. Install training & MLOps dependencies in Google Colab
!pip install -q lightgbm xgboost lifelines optuna mlflow pyarrow pydantic pyyaml onnx onnxmltools


In [ ]:
# 2. Mount Google Drive and configure workspace paths
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/cart-abandonment')
else:
    PROJECT_ROOT = Path('..').resolve()

DATA_ML_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml'
MLFLOW_URI = os.getenv('MLFLOW_TRACKING_URI', (PROJECT_ROOT / 'mlruns').as_uri())
print('DATA_ML_DIR:', DATA_ML_DIR)
print('MLFLOW_URI :', MLFLOW_URI)


In [ ]:
# 3. Configure MLflow Tracking Server / Drive Store
import mlflow

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('cart-abandon/ml/intent-classifier')


In [ ]:
# 4. Load Shared Feature Schema Contract & Parquet Splits
import lightgbm as lgb
import pandas as pd
import yaml
from sklearn.metrics import average_precision_score, roc_auc_score

with open(PROJECT_ROOT / 'configs' / 'feature_schema.yaml', 'r', encoding='utf-8') as f:
    schema = yaml.safe_load(f)

intent_spec = schema['models']['intent_classifier']
FEATURE_COLS = [feat['name'] for feat in intent_spec['features']]
TARGET_COL = intent_spec['target']['name']

train_df = pd.read_parquet(DATA_ML_DIR / 'intent_train.parquet')
eval_df = pd.read_parquet(DATA_ML_DIR / 'intent_eval.parquet')
print(f'Loaded train={len(train_df):,}, eval={len(eval_df):,}, features={len(FEATURE_COLS)}')


In [ ]:
# 5. Fit LightGBM Intent Classifier, Log to MLflow & Verify Blocking Quality Gate (AUC >= 0.80)
with open(PROJECT_ROOT / 'configs' / 'ml' / 'intent_classifier.yaml', 'r', encoding='utf-8') as f:
    model_cfg = yaml.safe_load(f)

params = dict(model_cfg['hyperparameters'])
early_stop = params.pop('early_stopping_rounds', 50)

with mlflow.start_run(run_name='lgbm-intent-baseline'):
    mlflow.log_params(params)
    clf = lgb.LGBMClassifier(**params)
    clf.fit(
        train_df[FEATURE_COLS],
        train_df[TARGET_COL],
        eval_set=[(eval_df[FEATURE_COLS], eval_df[TARGET_COL])],
        callbacks=[lgb.early_stopping(early_stop, verbose=50)],
    )
    preds = clf.predict_proba(eval_df[FEATURE_COLS])[:, 1]
    auc = roc_auc_score(eval_df[TARGET_COL], preds)
    pr_auc = average_precision_score(eval_df[TARGET_COL], preds)
    mlflow.log_metrics({'auc_roc': auc, 'pr_auc': pr_auc})
    mlflow.lightgbm.log_model(
        clf, artifact_path='model', registered_model_name='CartAbandon_IntentClassifier'
    )
    print(f'Eval AUC-ROC: {auc:.4f} | Passed Blocking Gate (>= 0.80): {auc >= 0.80}')
